# 🫡 Save the Model

We need to save this model so that we can use it from various locations, including other notebooks or the model server, so we upload it to an s3-compatible storage.

>NOTE: Don't run all the cells all-in-one shot without changing the cluster specific variables.

### Install the required packages and define a function for the upload

If `pip` gives an error, don't worry about it. Things will just run fine regardless.

In [ ]:
%pip -q install -U pip
%pip -q install model-registry

# 🤩 Kubeflow Registry

We need a metadata registry for storing information such as version, author, and model location of the models we are building.

We are using Kubeflow model registry as a canonical data source by storing such information.

Here are some reasons to use a registry (_from Kubeflow website_):

- Track models available on storage: once the model is stored, it can then be tracked in the Kubeflow Model Registry for managing its lifecycle. The Model Registry can catalog, list, index, share, record, organize this information. This allows the Data Scientist to compare different versions and revert to previous versions if needed.

- Track and compare performance: View key metrics like accuracy, recall, and precision for each model version. This helps identify the best-performing model for deployment.

- Create lineage: Capture the relationships between data, code, and models. This enables the Data Scientist to understand the origin of each model and reproduce specific experiments.

- Collaborate: Share models and experiment details with the MLOps Engineer for deployment preparation. This ensures a seamless transition from training to production.

An instance of the registry is available in your dev environment as well. 


# 🪣 S3 Storage

We use S3 storage as the backend for our Model Registry. This means our models are stored in S3, and the Model Registry keeps track of their locations. 

Thanks to the Kubeflow Model Registry, we can push our models to both our S3 storage and register them at the same time, making it easy to store and keep track of new models we produce 🕵️

In [ ]:
from model_registry import ModelRegistry
from model_registry.utils import S3Params
from model_registry.exceptions import StoreError
import os

‼️⚠️ IMPORTANT ⚠️‼️

Add your user name and cluster domain (apps.xxx) that are shared with you before. We need them for the model registry URL.

In [ ]:
# Add your user name and cluster domain (apps.xxx) that are shared with you before

username = "USER"
cluster_domain = "DOMAIN"

In [ ]:
# Set up the model registry connection
model_registry_url = f"https://registry-rest.{cluster_domain}"
author_name = username

print(model_registry_url)

registry = ModelRegistry(server_address=model_registry_url, port=443, author=author_name, is_secure=False)

In [ ]:
# Model details we want to register
registered_model_name = "jukebox"
s3_model_bucket = "models"
s3_model_prefix = f"{s3_model_bucket}/{registered_model_name}"
version = "2"
# local paths
model_path = f"models/jukebox/{version}"
onnx_model = f"{model_path}/onnx"
torch_model = f"{model_path}/torch"
artifacts = f"{model_path}/artifacts"
# remote S3 paths
s3_region = "us"
s3_onnx = f"{s3_model_prefix}/onnx/{registered_model_name}"
s3_torch = f"{s3_model_prefix}/torch/{registered_model_name}"
s3_artifacts = f"{s3_model_prefix}/artifacts/{registered_model_name}"
minio_endpoint = f"https://minio-s3-s3-minio-dev.{cluster_domain}"

#### 🙏 Thanks to data connections, S3 bucket credentials are available in the notebook!

We explicitly fetch the S3 bucket, but the others are used automagically behind the scenes when we upload and register our model 🧙‍♂️

In [ ]:
# upload parameters for s3 connections
s3_upload_params_onnx = S3Params(
    bucket_name=os.environ.get('AWS_S3_BUCKET'),
    s3_prefix=f"{s3_onnx}/{version}",
)
s3_upload_params_torch = S3Params(
    bucket_name=os.environ.get('AWS_S3_BUCKET'),
    s3_prefix=f"{s3_torch}/{version}",
)
s3_upload_params_artifacts = S3Params(
    bucket_name=os.environ.get('AWS_S3_BUCKET'),
    s3_prefix=f"{s3_artifacts}/{version}",
)

In [ ]:
# artifact update function
def update_artifact(model_name, model_version, new_uri, storage_path):
    artifact = registry.get_model_artifact(model_name, model_version)
    print(f"Got Artifact {artifact.name} with ID: {artifact.id}\n Current URI: {artifact.uri}\n Updating with URI: {new_uri}\n Current StoragePath: {artifact.storage_path}")
    artifact.uri = new_uri
    artifact.storage_path = storage_path
    registry.update(artifact)

In [ ]:
# upload function
def register(model_name, data_path,
             model_format_name, author, model_format_version,
             model_version, storage_path, description,
             metadata, upload_parms):
    try:
        # register onnx model
        registered_model = registry.upload_artifact_and_register_model(
            name=model_name,
            model_files_path=data_path,
            model_format_name=model_format_name,
            author=username,
            model_format_version=model_format_version,
            version=model_version,
            storage_path=storage_path,
            description=description,
            metadata=metadata,
            upload_params=upload_parms
        )
        print(f"'{model_name}' version '{model_version}'\n URL: https://rhods-dashboard-redhat-ods-applications.{cluster_domain}/modelRegistry/registry/registeredModels/1/versions/{registry.get_model_version(model_name, model_version).id}/details")
    except StoreError:
        stored_version = registry.get_model_version(registered_model_name, f"{version}-onnx")
        print(f"Model version {stored_version.name}-{stored_version.id} already exists: Updating URI...")
        new_uri = f"s3://{storage_path}?endpoint={minio_endpoint}&defaultRegion={s3_region}"
        update_artifact(model_name, model_version, new_uri, storage_path)

In [ ]:
# data to register in the model registry
models = [
    {
        "model_name": registered_model_name,
        "data_path": onnx_model,
        "author": username,
        "model_format_name": "onnx",
        "model_format_version": "1",
        "model_version": f"{version}-onnx",
        "storage_path": f"{s3_model_bucket}/{s3_onnx}",
        "description": "Dense Neural Network trained on music data (ONNX)",
        "metadata": {
                    "accuracy": 0.3,
                    "license": "apache-2.0"
                },
        "upload_parms": s3_upload_params_onnx
    },
    {
        "model_name": registered_model_name,
        "data_path": torch_model,
        "author": username,
        "model_format_name": "torch",
        "model_format_version": "1",
        "model_version": f"{version}-torch",
        "storage_path": f"{s3_model_bucket}/{s3_torch}",
        "description": "Dense Neural Network trained on music data (TORCH)",
        "metadata": {
                    "accuracy": 0.3,
                    "license": "apache-2.0"
                },
        "upload_parms": s3_upload_params_torch
    },
    {
        "model_name": registered_model_name,
        "data_path": artifacts,
        "author": username,
        "model_format_name": "pickle",
        "model_format_version": "1",
        "model_version": f"{version}-artifacts",
        "storage_path": f"{s3_model_bucket}/{s3_artifacts}",
        "description": "Companion artifacts (Pickle)",
        "metadata": {
                    "accuracy": 0.3,
                    "license": "apache-2.0"
                },
        "upload_parms": s3_upload_params_artifacts
    }
]

In [ ]:
# register models
for model in models:
    print(f"Registering: {model.get('model_version')}...")
    register(model_name=model.get('model_name'),
             data_path=model.get('data_path'),
             model_format_name=model.get('model_format_name'),
             author=model.get('author'),
             model_format_version=model.get('model_format_version'),
             model_version=model.get('model_version'),
             storage_path=model.get('storage_path'),
             description=model.get('description'),
             metadata=model.get('metadata'),
             upload_parms=model.get('upload_parms'))

### We can also get some nice information about our model from the Model Registry

In [ ]:
# Print the general info of registered model
model = registry.get_registered_model(registered_model_name)
print("Registered Model:", model.name, "with ID", model.id)

In [ ]:
# Print the version info of registered model
versions = registry.get_model_versions(registered_model_name)
for v in versions:
    print(f"Model Version: {v.name}, with ID {v.id}: state {v.state}")

In [ ]:
# Print the artifact info of registered model
for v in versions:
    art = registry.get_model_artifact(registered_model_name, v.name)
    print(f"Model Artifact ID {art.id}, URI: {art.uri}, storage path: {art.storage_path}")

### 🥁 Next Step

Now that you've saved the model to S3 storage & registry, you can refer to the model by using a data connection and serve the model as an API.

But first, we need to deploy the model using the Model Server